# ⚡ Realtime WebRTC Voice Agent (Kaggle GPU + Cloudflare Tunnel)

This notebook runs the **Full-Duplex WebRTC Voice Agent Server** on a Kaggle GPU (T4 / P100 / A100):
- 🎙️ **ASR**: Qwen3-ASR (0.6B) running in **FP16 CUDA** (~15ms chunk, <200ms decode)
- 🧠 **VAD**: Silero Neural VAD for high-precision voice vs. noise discrimination
- 💬 **LLM**: Ollama Gemma 4 (31B Cloud) with Tool Calling & Thinking Mode Disabled
- 🔊 **TTS**: Multi-Engine TTS (VITS Neural Local Hindi / Edge-TTS / Cartesia Sonic-3)
- 🌐 **WebRTC**: Real-time peer-to-peer audio streaming & DataChannel telemetry over **Cloudflare Quick Tunnel**

---

In [ ]:
# 1. Check GPU Status
!nvidia-smi

In [ ]:
# 2. Clone Repository & Navigate
!git clone https://github.com/yashNiwane/realtime-voice-agent-webrtc.git /kaggle/working/realtime-voice-agent-webrtc
%cd /kaggle/working/realtime-voice-agent-webrtc

In [ ]:
# 3. Install System Codecs & Python Dependencies
!apt-get update -qq && apt-get install -y -qq libavdevice-dev libavfilter-dev libopus-dev libvpx-dev pkg-config wget curl
!pip install -q -r requirements.txt

In [ ]:
# 4. Launch Cloudflare Tunnel & Print Public WebRTC Agent URL
import subprocess, time, re, os

if not os.path.exists('cloudflared'):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Start tunnel
!rm -f /tmp/tunnel.log
tunnel_proc = subprocess.Popen('./cloudflared tunnel --url http://localhost:7860 > /tmp/tunnel.log 2>&1', shell=True)

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists('/tmp/tunnel.log'):
        with open('/tmp/tunnel.log', 'r') as f:
            content = f.read()
            matches = re.findall(r'https://[-0-9a-z]*\.trycloudflare\.com', content)
            if matches:
                tunnel_url = matches[-1]
                break

print('='*70)
if tunnel_url:
    print(f'🚀 PUBLIC WEBRTC DASHBOARD URL: {tunnel_url}')
    print(f'📡 REST SIGNALING ENDPOINT:     {tunnel_url}/offer')
else:
    print('⚠️ Tunnel initializing. Check /tmp/tunnel.log')
print('='*70)

In [ ]:
# 5. Start Full-Duplex WebRTC Voice Agent Server (GPU Accelerated)
os.environ['DEVICE'] = 'cuda'
os.environ['TORCH_DTYPE'] = 'float16'
os.environ['PORT'] = '7860'
os.environ['HOST'] = '0.0.0.0'

!python -m server.webrtc_server